# Identificação e Correção de Dados Inválidos

## Estratégias para Identificar Dados Inválidos

### 1. Análise Exploratória Inicial

In [3]:
import pandas as pd
import numpy as np
import pandera.pandas as pa

# Carregando dados
df = pd.read_csv('dados.csv')

# Estatísticas básicas
print(df.describe())

# Verificando valores nulos
print(df.isnull().sum())

# Verificando tipos de dados
print(df.dtypes)

          idade      salario
count  50.00000    50.000000
mean   40.40000  2477.941604
std    17.88398  1492.588455
min    -1.00000  -500.000000
25%    35.25000  1570.100894
50%    45.50000  2309.607819
75%    50.75000  3687.221923
max    64.00000  4927.363553
idade      0
salario    0
email      0
dtype: int64
idade        int64
salario    float64
email       object
dtype: object


### 2. Validação com Pandera

In [4]:
# Definindo esquema de validação
schema = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0)),
    'email': pa.Column(str, checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'))
})

# Validando e capturando erros
try:
    schema.validate(df)
except pa.errors.SchemaError as e:
    print(f"Erros encontrados: {e}")

Erros encontrados: Column 'idade' failed element-wise validator number 0: greater_than_or_equal_to(0) failure cases: -1, -1, -1, -1, -1


## Técnicas de Correção de Dados

### 1. Tratamento de Valores Nulos

In [5]:
# Estratégias para valores nulos
def tratar_nulos(df):
    # Preencher com valor médio
    df['idade'] = df['idade'].fillna(df['idade'].mean())
    
    # Preencher com valor mais frequente
    df['departamento'] = df['departamento'].fillna(df['departamento'].mode()[0])
    
    # Preencher com valor específico
    df['status'] = df['status'].fillna('pendente')
    
    return df

### 2. Correção de Tipos de Dados

In [6]:
def corrigir_tipos(df):
    # Convertendo string para datetime
    df['data_nascimento'] = pd.to_datetime(df['data_nascimento'], errors='coerce')
    
    # Convertendo string para numérico
    df['salario'] = pd.to_numeric(df['salario'], errors='coerce')
    
    # Convertendo para categoria
    df['departamento'] = df['departamento'].astype('category')
    
    return df

### 3. Correção de Valores Inválidos

In [7]:
def corrigir_valores(df):
    # Substituindo valores negativos
    df.loc[df['idade'] < 0, 'idade'] = df['idade'].mean()
    
    # Corrigindo emails inválidos
    df.loc[~df['email'].str.contains('@', na=False), 'email'] = None
    
    # Limitando valores extremos
    df.loc[df['salario'] > 1000000, 'salario'] = 1000000
    
    return df

## Pipeline de Validação e Correção

In [8]:
def pipeline_validacao_correcao(df):
    # 1. Análise inicial
    print("Análise inicial:")
    print(df.info())
    
    # 2. Tratamento de nulos
    df = tratar_nulos(df)
    
    # 3. Correção de tipos
    df = corrigir_tipos(df)
    
    # 4. Correção de valores
    df = corrigir_valores(df)
    
    # 5. Validação final
    try:
        schema.validate(df)
        print("Dados validados com sucesso!")
    except pa.errors.SchemaError as e:
        print(f"Ainda existem erros: {e}")
    
    return df

### Exemplo Prático Completo

In [10]:
import pandas as pd
import pandera.pandas as pa
import numpy as np

# Criando dados de exemplo com problemas
dados = {
    'id': [1, 2, 3, 4, 5],
    'nome': ['João', 'Maria', 'Pedro', None, 'Ana'],
    'idade': [25, -30, 35, 40, np.nan],
    'salario': ['5000', '6000', '7000', '8000', '9000'],
    'email': ['joao@email.com', 'maria@email', 'pedro@email.com', None, 'ana@email.com'],
    'departamento': ['TI', 'RH', 'Vendas', 'TI', None]
}

df = pd.DataFrame(dados)

# Definindo esquema de validação
schema = pa.DataFrameSchema({
    'id': pa.Column(int, checks=pa.Check.gt(0)),
    'nome': pa.Column(str, nullable=False),
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0)),
    'email': pa.Column(str, checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'), nullable=True),
    'departamento': pa.Column(str, checks=pa.Check.isin(['TI', 'RH', 'Vendas', 'Financeiro']), nullable=True)
})

# Aplicando pipeline de correção
try:
    df_corrigido = pipeline_validacao_correcao(df)

    # Verificando resultados
    print("\nDados após correção:")
    print(df_corrigido)
except KeyError as e:
    print(f"Erro de chave: {e}")
    print("Verifique se todas as colunas necessárias estão presentes no DataFrame")

Análise inicial:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            5 non-null      int64  
 1   nome          4 non-null      object 
 2   idade         4 non-null      float64
 3   salario       5 non-null      object 
 4   email         4 non-null      object 
 5   departamento  4 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 368.0+ bytes
None
Erro de chave: 'status'
Verifique se todas as colunas necessárias estão presentes no DataFrame


# Boas Práticas para Correção de Dados

Documente as Alterações: Mantenha um registro das correções realizadas

Preserve os Dados Originais: Sempre trabalhe com cópias dos dados

Valide Após Cada Etapa: Verifique se as correções não introduziram novos problemas

Use Métricas de Qualidade: Defina e monitore métricas de qualidade dos dados

Automatize o Processo: Crie pipelines reprodutíveis para validação e correção

# Tratamento de Casos Especiais

Dados Sensíveis: Tenha cuidado ao corrigir dados pessoais ou sensíveis

Dados Temporais: Considere a sazonalidade e tendências temporais

Dados Categóricos: Mantenha a consistência nas categorias

Dados Relacionais: Preserve a integridade referencial entre tabelas

Dados em Lote: Implemente estratégias eficientes para grandes volumes de dados